# Drive, github and requirements settings

In [ ]:
from google.colab import drive
from google.colab import userdata


drive.mount('/content/drive')
KEY = userdata.get('KEY')

Mount colab to google drive

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount = True)

Mounted at /content/drive


set Github access token

In [4]:
import os
from google.colab import userdata
github_access_token = userdata.get('GITHUB_TOKEN')

# Replace 'your_token_here' with your actual token and 'your_repo_url_here' with your repository URL
os.environ['GITHUB_TOKEN'] = github_access_token

repo_url = 'https://github.com/sustaz/principle_of_law_detection.git'
modified_url = repo_url.replace('https://', f'https://{os.environ["GITHUB_TOKEN"]}@')

Clone Repository

In [2]:
!git clone {modified_url}

Cloning into 'principle_of_law_detection'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (21/21), done.
remote: Total 26 (delta 9), reused 18 (delta 4), pack-reused 0
Receiving objects: 100% (26/26), 96.36 KiB | 1.69 MiB/s, done.
Resolving deltas: 100% (9/9), done.


Set github credentials

In [6]:
!chmod +x ./principle_of_law_detection/bash_commands/set_git_credentials.sh
!./principle_of_law_detection/bash_commands/set_git_credentials.sh

Syncronize notebook version

In [ ]:
!cp /content/drive/MyDrive/POLINE/gpt_notebook.ipynb /content/principle_of_law_detection/

In [ ]:
!cp /content/drive/MyDrive/POLINE/old_test_outputs.ipynb /content/principle_of_law_detection/

Add, commit, push code

In [7]:
!chmod +x ./principle_of_law_detection/bash_commands/add_commit_push.sh
!./principle_of_law_detection/bash_commands/add_commit_push.sh "cleaned gpt notebook and built library"

[main 2616a1d] cleaned gpt notebook and built library
 4 files changed, 232 insertions(+), 1 deletion(-)
 create mode 100644 utils/evaluation.py
 create mode 100644 utils/save_results.py
 create mode 100644 utils/utils.py
Enumerating objects: 10, done.
Counting objects: 100% (10/10), done.
Delta compression using up to 2 threads
Compressing objects: 100% (7/7), done.
Writing objects: 100% (7/7), 2.69 KiB | 2.69 MiB/s, done.
Total 7 (delta 3), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (3/3), completed with 2 local objects.
To https://github.com/sustaz/principle_of_law_detection.git
   27054d3..2616a1d  main -> main


Install requirements

In [2]:
!pip install -r '/content/principle_of_law_detection/requirements.txt'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.4/327.4 kB 8.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 8.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 10.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 6.4 MB/s eta 0:00:00


# Import libraries

In [5]:
from principle_of_law_detection.utils import gpt_utils as gu, save_results as sr, text_preprocessing as tp, evaluation as ev
import json
import os
import pandas as pd

# MASSIVE EXPERIMENTS

In [ ]:
def jpol_prompt(txt):
  prompt =  f"""" Extract portions of the text from the argumentative part of the judgment that fit the definition of a JPOL.

    {txt}

    A JPOL (Judicial Principle of Law) should:

    Source and Content:
        Be a portion of text extracted from the argumentative part of a judgment.
        Contain the interpretation provided by the deciding court or an endorsed interpretation from a previous decision.

    Nature of Interpretation:
        Interpretate a rule, a general principle, or the consequences stemming from the application of a rule or principle within the legal system.
        Not be a rephrase of the legislation.

    Citations and Endorsements:
        A JPOL can cite another JPOL.
        A JPOL can endorse a precedent of a European Court or an Advocate General's statement.

    Use the following format for the output:
      Paragraph number: Y if JPOL.
      Paragraph number: N if not JPOL."""

  return prompt



system_prompt = "You are a professionist judge with strong knowledge on jurisdiction and tax law, expert about Judicial Principles of Law (JPOLs) from legal judgments. "

In [6]:
jsons_root = "/content/drive/MyDrive/POLINE/JSON TAG/poline_jsons/"
annotations_files = os.listdir(jsons_root)

In [ ]:
judements_root = "/content/drive/MyDrive/POLINE/Dataset_V1/Judgements_Subset"
judgemnts = os.listdir(judements_root)

## Call massive GPT - PAY ATTENTION TO THIS CELL

In [ ]:
prompt_name = "p_aspries_2"
responses = []

for idx, judgement in enumerate(judgemnts):


  with open(os.path.join(judements_root, judgement)) as f:
    file = f.read()

  txt = "".join(file)

  txt = tp.extract_text_between_markers(txt)

  #write_text_to_docx(txt, f"/content/drive/MyDrive/POLINE/Dataset_V1/Preprocessed_Judgements_Subset/{judgement}.docx")

  response = gu.ask_gpt_2(jpol_prompt(txt), system_prompt, model="gpt-4o", max_tokens=4000, temperature=0.2, top_p=1)

  sr.write_text_to_docx(response, f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/{prompt_name}_{judgement}_full_response.docx")

  responses.append((response, judgement))

Save results

In [ ]:
dfs = []
for response, file_name in responses:
    dfs.append((sr.extract_paragraphs(response), file_name[:5]))

results_df = sr.concatenate_dataframes(dfs)
results_df.to_excel(f"/content/drive/MyDrive/POLINE/Results/preprocessed_input/{prompt_name}.xlsx", index=False)

Read GT

In [ ]:
ground_truths = []

for ann_file in annotations_files:

  ann_dict = json.load(open(os.path.join(jsons_root,ann_file)))

  for ann in ann_dict['annotations']:
    paragraph_number = ann['text'].split()[0]
    label = ann['type']
    file_name = ann_file[:5]

    ground_truths.append((file_name, paragraph_number, label))

ground_truth_df = pd.DataFrame(ground_truths, columns=['file_name', 'paragraph_number', 'ground_truth'])

In [ ]:
ground_truth_df

,file_name,paragraph_number,ground_truth
0,ELVOS,25,JPOL
1,ELVOS,27,JPOL
2,ELVOS,28,JPOL
3,ELVOS,29,JPOL
4,ELVOS,31,JPOL
...,...,...,...
99,Micha,53,JPOL
100,Micha,55,JPOL
101,Micha,55,JPOL
102,Micha,68,JPOL


# EVALUATION

In [ ]:
results_df = pd.read_excel("/content/drive/MyDrive/POLINE/Results/preprocessed_input/p_aspries_2.xlsx")

# Chiamare la funzione
metrics_by_filename, merged_df = compute_metrics(ground_truth_df, results_df)

# Mostrare i risultati
print(metrics_by_filename)

precision, recall, f1 = compute_total_metrics(ground_truth_df, results_df)

print(f'<--------------------->')

print(f'Total Precision: {precision:.2f}')
print(f'Total Recall: {recall:.2f}')
print(f'Total F1-Score: {f1:.2f}')

  file_name  precision    recall        f1
0     A & G   1.000000  1.000000  1.000000
1     Autor   0.700000  0.777778  0.736842
2     Boehr   0.608696  1.000000  0.756757
3     DNB B   0.928571  0.812500  0.866667
4     ELVOS   0.533333  1.000000  0.695652
5     Finan   0.600000  0.937500  0.731707
6     Micha   1.000000  0.687500  0.814815
7     Minis   0.823529  1.000000  0.903226
<--------------------->
Total Precision: 0.74
Total Recall: 0.89
Total F1-Score: 0.81


- baseline
- estendere dataset con i 3 file mancanti
- scrivere i risultati dell'evaluation su excel in drive

